# 🗄️ La práctica de bases de datos — dos tablas, una llave, cuatro gemelas

> 📽 *Vienes de las láminas BD-01 a BD-03: dónde viven los datos, el modelo de datos y SQL.*

Dos ciudades, dos años de clima, una base de datos que ES un archivo — y cuatro preguntas
que se contestan por dos caminos (SQL y pandas) y tienen que dar **lo mismo**.

In [1]:
# ✅ Punto de control
import os
try:
    import pandas as pd, sqlite3
    print("✅ Entorno listo · pandas", pd.__version__)
except ImportError as e:
    print("⏳ Falta un paquete:", e, "— ¿kernel .venv?")
for f in ("clima_cdmx.csv", "clima_hermosillo.csv"):
    print(("✅" if os.path.exists(f) else "⏳ falta"), f)
print("(clima_hermosillo.csv viene en el reparto de hoy; clima_cdmx.csv es tu descarga de la práctica de APIs)")

✅ Entorno listo · pandas 2.2.2
✅ clima_cdmx.csv
✅ clima_hermosillo.csv
(clima_hermosillo.csv viene en el reparto de hoy; clima_cdmx.csv es tu descarga de la práctica de APIs)


## 1 · Construir la base: dos tablas, una llave

El modelo de datos de la lámina BD-03, en código. Fíjate en dos cosas: la **llave**
(`ciudad_id`) y el **renombrado** — la API entrega nombres crudos y larguísimos
(`temperature_2m_max`…); al construir la base les ponemos nombres cortos en español.
Limpiar nombres ES parte del trabajo de datos.

In [2]:
import pandas as pd
import sqlite3

ciudades = pd.DataFrame({
    "ciudad_id": [1, 2],
    "nombre": ["CDMX", "Hermosillo"],
    "latitud": [19.43, 29.07], "longitud": [-99.13, -110.96],
})

RENOMBRES = {                       # crudo de la API  →  nombre del curso
    "temperature_2m_max": "t_max", "temperature_2m_min": "t_min",
    "temperature_2m_mean": "t_media", "precipitation_sum": "lluvia_mm",
    "windspeed_10m_max": "viento", "shortwave_radiation_sum": "radiacion",
}
cdmx = pd.read_csv("clima_cdmx.csv").rename(columns=RENOMBRES);        cdmx["ciudad_id"] = 1
hmo = pd.read_csv("clima_hermosillo.csv").rename(columns=RENOMBRES);   hmo["ciudad_id"] = 2
clima = pd.concat([cdmx, hmo], ignore_index=True)
print("columnas de la tabla clima:", list(clima.columns))

con = sqlite3.connect("clima.sqlite")        # una BD que ES un archivo — sin servidor
ciudades.to_sql("ciudades", con, if_exists="replace", index=False)
clima.to_sql("clima", con, if_exists="replace", index=False)
print("✅ clima.sqlite ·", len(ciudades), "ciudades ·", len(clima), "días")   # 🔮 DEBES VER: 2 y 1462

columnas de la tabla clima: ['fecha', 't_max', 't_min', 't_media', 'lluvia_mm', 'viento', 'radiacion', 'ciudad_id']
✅ clima.sqlite · 2 ciudades · 1462 días


## 🔎 ¿Qué acaba de pasar?

* **`sqlite3.connect("clima.sqlite")` no se conectó a ningún servidor: abrió un archivo** — y como
  no existía, lo creó. SQLite es un motor de base de datos que corre *dentro* de tu Python (viene
  con él), y la base entera es ese archivo. `con` es la conexión: el cable por el que le mandas SQL
  y te devuelve tablas.
* **`to_sql` copió cada DataFrame al archivo como una tabla:** `ciudades` (2 filas) y `clima`
  (1 462). `if_exists="replace"` deja volver a correr la celda sin error (rehace la tabla);
  `index=False` no guarda el índice 0, 1, 2… como columna.
* **«clima» nombra tres cosas distintas:** la *variable* de Python `clima` (un DataFrame en la
  memoria) · la *tabla* `clima` dentro de la base (la que lee `FROM clima`) · el *archivo*
  `clima.sqlite` (la base completa, con sus dos tablas).
* **Desde aquí hay dos caminos a los mismos datos:** pandas lee la variable; SQL, la tabla del
  archivo. Por eso las gemelas tienen que dar lo mismo.

## 2 · Lo regenerable no se versiona

`clima.sqlite` se reconstruye con la celda de arriba en dos segundos: al repo va **la receta
(este cuaderno), no el pastel**. Tu `.gitignore` todavía no lo sabe — esta celda le agrega dos
renglones (ábrelo después en VSCode y míralos).

In [3]:
import pathlib
gi = pathlib.Path(".gitignore")
lineas = gi.read_text().splitlines() if gi.exists() else []
for patron in ("*.sqlite", "informe.html"):      # la base de hoy · el informe de EDA de la próxima práctica
    if patron not in lineas:
        lineas.append(patron)
gi.write_text("\n".join(lineas) + "\n")
print("✅ .gitignore:", lineas)
# 🔮 DEBES VER al correr git status: clima.sqlite NO aparece — el cuaderno sí

✅ .gitignore: ['*.sqlite', 'informe.html']


## 3 · Las consultas gemelas — SQL y pandas deben dar LO MISMO

Las dos primeras vienen resueltas. Córrelas por los dos caminos y compara.

In [4]:
# Gemela 1 (resuelta) · ¿qué días llovió MUY fuerte (> 30 mm)?
print(pd.read_sql("SELECT fecha, ciudad_id, lluvia_mm FROM clima WHERE lluvia_mm > 30", con))
print()
print(clima.loc[clima.lluvia_mm > 30, ["fecha", "ciudad_id", "lluvia_mm"]].to_string(index=False))
# 🔮 DEBES VER: las mismas 12 filas por los dos caminos — 6 de CDMX (ciudad 1) y 6 de Hermosillo (ciudad 2);
#    la más brava: 67.7 mm en Hermosillo, el 5-sep-2025

         fecha  ciudad_id  lluvia_mm
0   2024-06-30          1       46.6
1   2024-08-16          1       31.7
2   2025-06-29          1       54.6
3   2025-09-17          1       30.4
4   2025-10-08          1       32.5
5   2024-01-23          2       46.1
6   2025-09-03          2       30.9
7   2025-09-05          2       67.7
8   2025-09-06          2       54.9
9   2025-10-11          2       41.3
10  2025-10-12          2       33.1

     fecha  ciudad_id  lluvia_mm
2024-06-30          1       46.6
2024-08-16          1       31.7
2025-06-29          1       54.6
2025-09-17          1       30.4
2025-10-08          1       32.5
2024-01-23          2       46.1
2025-09-03          2       30.9
2025-09-05          2       67.7
2025-09-06          2       54.9
2025-10-11          2       41.3
2025-10-12          2       33.1


In [5]:
# Gemela 2 (resuelta) · temperatura máxima promedio POR CIUDAD
print(pd.read_sql("SELECT ciudad_id, AVG(t_max) FROM clima GROUP BY ciudad_id", con))
print()
print(clima.groupby("ciudad_id")["t_max"].mean())
# 🔮 DEBES VER: 24.7 y 32.5 — el desierto se nota

   ciudad_id  AVG(t_max)
0          1   24.741997
1          2   32.469494

ciudad_id
1    24.741997
2    32.469494
Name: t_max, dtype: float64


In [ ]:
# Gemela 3 (TUYA) · ¿cuántos días CON lluvia (> 0 mm) tiene cada ciudad?
# ✏️ escribe la versión SQL y la de pandas; deben dar 401 (CDMX) y 132 (Hermosillo)
# pista SQL:    COUNT(*) cuenta las filas de cada grupo, como AVG(t_max) las promedia en la gemela 2
# pista pandas: .size() cuenta las filas de cada grupo de un groupby



In [ ]:
# Gemela 4 (TUYA, la difícil) · lo mismo, pero con el NOMBRE de la ciudad — exige el JOIN
# pista SQL:    ... FROM clima JOIN ciudades ON clima.ciudad_id = ciudades.ciudad_id ...
# pista pandas: clima.merge(ciudades, on="ciudad_id") y luego groupby("nombre")


## ▶ El cierre

En la terminal, el ciclo de siempre (+ `Fixes <TU-ID>` en el mensaje si abriste un issue en Linear para esta práctica):

```
git status          # 🔮 clima.sqlite NO aparece: lo regenerable no se versiona
git add .
git commit -m "Práctica de SQL: dos tablas, cuatro gemelas"
git push
```

**Si las gemelas 3 y 4 no salieron en clase, son la tarea** — el JOIN es la difícil y está bien
que duela. *La próxima práctica lee ESTA base desde otro cuaderno, la mira con lupa (EDA) y
entrena el modelo.*